# TxSON MET Data Cleaning Pipeline — Part 1: Processing

Standardized quality control for all TxSON meteorological stations.

This is **Part 1 of 2**. It runs the prewash + QC stages and writes every
result to `OUTPUT_DIR` as CSV. Part 2 (`txson_met_pipeline_2_dashboard.ipynb`)
reads those CSVs back in and builds the interactive dashboard — it does not
need to be run in the same session, only after this notebook has completed
at least once.

**Stages covered here:**
1. Mount Drive + discover `.dat` files
2. Prewash: `dup_cleaner → treat_subhourly_data → fill_missing_timestamps → **find_and_replace_wrong_data*`
  - `find_and_replace_wrong_data` funciton is currently commented out from this file since it serves duplicated function
3. Split valid / flagged RH
4. Stuck-sensor detection (multi-variable evidence scoring)
5. Gap categorization
6. Monthly averages
7. Jump detection (hourly + monthly)

**Not covered here:** Stage 8, the interactive Plotly/ipywidgets dashboard —
see the companion notebook.

**Requirements:** `pandas`, `numpy`, `scipy` (all preinstalled on Colab), and
`prewash_df.py` present in `SCRIPTS_DIR`.


In [6]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy import stats as scipy_stats
import glob, os, re, calendar, warnings
warnings.filterwarnings('ignore')
print('Libraries ready.')


Libraries ready.


## Step 1 — Mount Drive & configure paths
Set `DAT_DIR` to the folder containing your raw `*_met.dat` files.
`SCRIPTS_DIR` should point to the folder containing `prewash_df.py`.
`OUTPUT_DIR` is where every CSV this notebook produces gets written — **Part 2
reads from this same folder**, so keep it consistent across runs.


In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ▶ Edit these paths
DAT_DIR     = '/content/drive/MyDrive/TXSoil-DataFiles'
SCRIPTS_DIR = '/content/drive/MyDrive/DSResearch_TexasSoil2026'
OUTPUT_DIR  = '/content/drive/MyDrive/DSResearch_TexasSoil2026/output'

# Drive's FUSE mount can lag a few seconds after drive.mount() finishes,
# especially for folders shared across accounts - os.path.exists() can
# wrongly return False during that window. If SCRIPTS_DIR/DAT_DIR aren't
# actually visible yet, os.makedirs below would silently create a brand
# new duplicate folder with the same name instead of finding the real one.
# Fail loudly instead of doing that.
assert os.path.isdir(SCRIPTS_DIR), (
    f'SCRIPTS_DIR not visible yet: {SCRIPTS_DIR}\n'
    f'Wait a few seconds for Drive to finish syncing (common right after '
    f'mount, especially for folders shared across accounts) and re-run this '
    f'cell. Do not re-run the rest of the notebook until this passes - '
    f'os.makedirs would otherwise create a duplicate folder here.'
)
assert os.path.isdir(DAT_DIR), f'DAT_DIR not visible yet: {DAT_DIR}'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Column names (original TxSON met names, post prewash rename mapping) ─────
COL_TS   = 'Date'          # prewash renames TIMESTAMP → Date
COL_RH   = 'RH'
COL_TEMP = 'AirTC_Avg'     # prewash renames to Tair — we map back below
COL_RAIN = 'Rain_mm_Tot'   # prewash renames to Ppt — we map back below
COL_WIND = 'WS_ms_S_WVT'  # prewash renames to Wind speed — we map back

# Prewash output names → original TxSON met names
# Applied after prewash so stage 2 / stuck-sensor detection works unchanged
PREWASH_TO_ORIGINAL = {
    'Ppt':            'Rain_mm_Tot',
    'Tair':           'AirTC_Avg',
    'Wind speed':     'WS_ms_S_WVT',
    'Wind direction': 'WindDir_D1_WVT',
    'Srad':           'SlrW_Avg',
}

MONTHS       = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
MONTH_LABELS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

def infer_station(fpath):
    stem = os.path.basename(fpath).rsplit('.',1)[0]
    for pat in [r'[_\-]met$', r'^(valid|flagged|clean)[_\-]',
                r'[_\-]rh[_\-]monthly.*', r'[_\-]monthly.*',
                r'[_\-]gap[_\-]summary$', r'[_\-]prewashed$']:
        stem = re.sub(pat, '', stem, flags=re.I)
    return stem.strip('_-')

dat_files = sorted(glob.glob(os.path.join(DAT_DIR, '*_met.dat')))
print(f'Found {len(dat_files)} .dat files:')
for f in dat_files:
    print(f'  {os.path.basename(f)} → "{infer_station(f)}"')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 6 .dat files:
  CB01_met.dat → "CB01"
  CB04_met.dat → "CB04"
  CB06_met.dat → "CB06"
  FD02_met.dat → "FD02"
  FD03_met.dat → "FD03"
  WC05_met.dat → "WC05"


## Step 2 — Pipeline parameters
Single source of truth. All downstream cells read from here.


In [8]:
# ── PARAMETERS ───────────────────────────────────────────────────────────────

# Stage 3: Range filter
RH_MIN = 0.0
RH_MAX = 100.0

# Stage 4: Stuck-sensor evidence scoring
# These thresholds are tunable starting points, not derived from a formal
# calibration run. Adjust STUCK_RH_STD / STUCK_CORR_MIN if you see too many
# false positives (real stable-humidity periods) or missed dropouts.
STUCK_WINDOW_H   = 24     # one full diurnal cycle
STUCK_RH_STD     = 2.0    # +3 pts — RH barely moves over the window
STUCK_CORR_MIN   = 0.177  # +2 pts — RH stops tracking temperature (decoupled)
SUSPECT_SCORE    = 3
STUCK_SCORE      = 5

# Stage 5: Gap categories (hours)
GAP_CATS = [
    ('Short',     0,    24),
    ('Medium',   24,   168),
    ('Long',    168,   720),
    ('VeryLong', 720, None),
]

# Stage 6: Monthly averages
MIN_MONTH_COVERAGE = 0.6   # months below this fraction of valid hours → NaN

# Stage 7: Jump detection
# Within-year month-to-month thresholds (RH%) — data-derived
WITHIN_THRESH = {
    'Jan':22,'Feb':20,'Mar':20,'Apr':18,'May':18,'Jun':18,
    'Jul':18,'Aug':18,'Sep':20,'Oct':20,'Nov':20,'Dec':22,
}
CROSS_YEAR_Z  = 2.0    # z-score threshold for same-month cross-year anomaly
HOURLY_JUMP_PCT = 95   # percentile of hourly |ΔRH| used as hourly jump threshold

print('Parameters set.')


Parameters set.


## Step 3 — Load prewash functions
Imports directly from `prewash_df.py` — no CLI, no argparse conflict.

If you edit `prewash_df.py` and re-run this cell, it force-reloads the module
so your changes actually take effect — otherwise Colab keeps using whatever
version was in memory from the first import of the session. If this cell ever
fails, it prints exactly what's in `SCRIPTS_DIR` so a wrong path is obvious
immediately instead of a bare `ModuleNotFoundError`.


In [9]:
import sys, os, importlib

if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

if not os.path.exists(os.path.join(SCRIPTS_DIR, 'prewash_df.py')):
    raise FileNotFoundError(
        f'prewash_df.py not found in SCRIPTS_DIR: {SCRIPTS_DIR}\n'
        f'Contents of that folder: {os.listdir(SCRIPTS_DIR) if os.path.isdir(SCRIPTS_DIR) else "(folder does not exist)"}'
    )

# Force a reload if prewash_df is already cached in this session - otherwise
# editing prewash_df.py on Drive and re-running this cell silently keeps
# using the OLD in-memory version. This is the #1 cause of "I fixed the bug
# but it's still happening" when iterating on prewash_df.py.
import prewash_df
importlib.reload(prewash_df)

from prewash_df import (
    determine_data_file,
    file_to_indexed_df,
    dup_cleaner,
    treat_subhourly_data,
    fill_missing_timestamps,
    #find_and_replace_wrong_data,
    run_prewash_step,
)

print(f'Prewash functions imported from: {prewash_df.__file__}')


Prewash functions imported from: /content/drive/MyDrive/DSResearch_TexasSoil2026/prewash_df.py


## Stage 2 — Prewash
`dup_cleaner → treat_subhourly_data → fill_missing_timestamps → find_and_replace_wrong_data`

Each step auto-writes a diff report CSV to `OUTPUT_DIR`. Output is renamed back to
original TxSON column names for downstream compatibility.

**Writes:** `{station}_met_prewashed.csv`, plus one `*_report.csv` per step.


In [10]:
# ── Stage 2: Prewash all met .dat files ─────────────────────────────────────
PREWASHED = {}   # station_id → prewashed df (original column names)

PREWASH_STEPS = [
    dup_cleaner,
    treat_subhourly_data,
    fill_missing_timestamps,
    #find_and_replace_wrong_data,
]

for fpath in dat_files:
    sid       = infer_station(fpath)
    data_type = determine_data_file(fpath)
    out_path  = os.path.join(OUTPUT_DIR, f'{sid}_met_prewashed.csv')

    if data_type == 'unknown':
        print(f'  {sid}: unknown file type — skipping')
        continue

    print(f'\n{sid}:')
    df = file_to_indexed_df(fpath, data_type)
    if df is None:
        print(f'  Could not read file — skipping')
        continue

    for step in PREWASH_STEPS:
        df = run_prewash_step(step, df, out_path)

    # Map prewash renamed columns back to original TxSON met names
    df = df.rename(columns=PREWASH_TO_ORIGINAL)

    PREWASHED[sid] = df
    df.to_csv(out_path)
    print(f'  Saved: {os.path.basename(out_path)}  ({len(df):,} rows)')

print(f'\nPrewash complete. Stations: {list(PREWASHED.keys())}')



CB01:
  Saved: CB01_met_prewashed.csv  (15,597 rows)

CB04:
  Saved: CB04_met_prewashed.csv  (98,907 rows)

CB06:
  Saved: CB06_met_prewashed.csv  (74,596 rows)

FD02:
  Saved: FD02_met_prewashed.csv  (99,914 rows)

FD03:
  Saved: FD03_met_prewashed.csv  (86,211 rows)

WC05:
  Saved: WC05_met_prewashed.csv  (62,874 rows)

Prewash complete. Stations: ['CB01', 'CB04', 'CB06', 'FD02', 'FD03', 'WC05']


## Stage 3 — Split valid / flagged RH
Splits each prewashed file into valid and flagged CSVs based on RH range.
Original values preserved in flagged file.

**Writes:** `valid_{station}_met.csv`, `flagged_{station}_met.csv`.


In [11]:
# ── Stage 3: Split valid / flagged ──────────────────────────────────────────
VALID_HOURLY = {}   # station → valid hourly df

for sid, df in PREWASHED.items():
    if COL_RH not in df.columns:
        print(f'  {sid}: no RH column found')
        continue

    rh        = pd.to_numeric(df[COL_RH], errors='coerce')
    in_range  = rh.between(RH_MIN, RH_MAX, inclusive='both') & rh.notna()
    is_null   = rh.isna()

    # NaN rows (e.g. from fill_missing_timestamps) are real gaps, not bad
    # readings - keep them in the valid frame so Stage 5 can catalog them.
    # Only out-of-range *numeric* values go to the flagged frame.
    df_valid   = df[in_range | is_null].copy()
    df_flagged = df[~in_range & ~is_null].copy()

    VALID_HOURLY[sid] = df_valid

    vpath = os.path.join(OUTPUT_DIR, f'valid_{sid}_met.csv')
    fpath = os.path.join(OUTPUT_DIR, f'flagged_{sid}_met.csv')
    df_valid.to_csv(vpath)
    df_flagged.to_csv(fpath)

    print(f'  {sid}: {len(df_valid):,} valid  |  {len(df_flagged):,} flagged  '
          f'({len(df_flagged)/len(df)*100:.1f}% flagged)')

print('\nStage 3 complete.')


  CB01: 15,597 valid  |  0 flagged  (0.0% flagged)
  CB04: 83,595 valid  |  15,312 flagged  (15.5% flagged)
  CB06: 74,517 valid  |  79 flagged  (0.1% flagged)
  FD02: 99,913 valid  |  1 flagged  (0.0% flagged)
  FD03: 79,742 valid  |  6,469 flagged  (7.5% flagged)
  WC05: 62,874 valid  |  0 flagged  (0.0% flagged)

Stage 3 complete.


## Stage 4 — Stuck-sensor detection
Multi-variable evidence scoring on valid hourly data. Results written to individual
CSV per station + combined file.

**Writes:** `audit_stuck_{station}.csv`, `audit_stuck_all_stations.csv`, and
**`clean_{station}_met.csv`** — the RH-nulled dataframe (`VALID_CLEAN`), which the
dashboard notebook reloads directly instead of recomputing this stage.


In [12]:
# ── Stage 4: Stuck-sensor detection ─────────────────────────────────────────
def categorize_gap_duration(hours):
    for name, lo, hi in GAP_CATS:
        if hi is None or hours < hi:
            return name
    return 'VeryLong'


def detect_stuck_sensor(df, sid):
    rh    = df[COL_RH].copy() if COL_RH in df.columns else None
    if rh is None:
        return df.copy(), pd.DataFrame()

    w     = STUCK_WINDOW_H
    score = pd.Series(0.0, index=df.index)

    # +3: RH rolling std (primary signal)
    rh_std  = rh.rolling(w, min_periods=w//2).std()
    score  += np.where(rh_std < STUCK_RH_STD, 3, 0)

    # +2: |RH-Temp rolling correlation| (diurnal decoupling)
    roll_corr = None
    if COL_TEMP in df.columns:
        roll_corr = rh.rolling(w, min_periods=w//2).corr(df[COL_TEMP]).abs()
        score    += np.where(roll_corr < STUCK_CORR_MIN, 2, 0)

    flag_series = pd.Series('OK', index=df.index)
    flag_series[score >= SUSPECT_SCORE] = 'SUSPECT'
    flag_series[score >= STUCK_SCORE]   = 'STUCK'

    flagged = flag_series.isin(['SUSPECT','STUCK'])
    run_id  = (flagged != flagged.shift()).cumsum()

    df_clean = df.copy()
    audit    = []

    for rid, grp in df[flagged].groupby(run_id[flagged]):
        duration_h = len(grp)
        flag_type  = flag_series[grp.index].value_counts().idxmax()
        orig_rh    = rh[grp.index].values
        mid        = grp.index[len(grp)//2]

        factors = []
        std_val = float(rh_std.reindex([mid]).iloc[0]) if mid in rh_std.index else np.nan
        if not np.isnan(std_val) and std_val < STUCK_RH_STD:
            factors.append(f'RH_std={std_val:.3f}<{STUCK_RH_STD}')
        if roll_corr is not None:
            corr_val = float(roll_corr.reindex([mid]).iloc[0]) if mid in roll_corr.index else np.nan
            if not np.isnan(corr_val) and corr_val < STUCK_CORR_MIN:
                factors.append(f'RH_Temp_corr={corr_val:.3f}<{STUCK_CORR_MIN}')

        audit.append({
            'station':       sid,
            'start':         grp.index[0],
            'end':           grp.index[-1],
            'duration_h':    duration_h,
            'gap_category':  categorize_gap_duration(duration_h),
            'flag':          flag_type,
            'avg_score':     round(score[grp.index].mean(), 2),
            'orig_rh_mean':  round(np.nanmean(orig_rh), 3),
            'orig_rh_std':   round(np.nanstd(orig_rh),  3),
            'orig_rh_min':   round(np.nanmin(orig_rh),  3),
            'orig_rh_max':   round(np.nanmax(orig_rh),  3),
            'factors':       ' | '.join(factors),
        })

        df_clean.loc[grp.index, COL_RH] = np.nan

    return df_clean, pd.DataFrame(audit)


# ── Run detection + save per-station CSVs ────────────────────────────────────
VALID_CLEAN  = {}
ALL_STUCK    = []

for sid, df in VALID_HOURLY.items():
    df_clean, audit = detect_stuck_sensor(df, sid)
    VALID_CLEAN[sid] = df_clean

    # persist the cleaned frame so the dashboard notebook can reload it directly
    df_clean.to_csv(os.path.join(OUTPUT_DIR, f'clean_{sid}_met.csv'))

    n_periods = len(audit)
    n_hours   = int(audit['duration_h'].sum()) if not audit.empty else 0
    print(f'{sid}: {n_periods} stuck period(s), {n_hours:,} hours nulled')

    if not audit.empty:
        # Per-station file
        path = os.path.join(OUTPUT_DIR, f'audit_stuck_{sid}.csv')
        audit.to_csv(path, index=False)
        print(f'  Saved: audit_stuck_{sid}.csv')

        # Print summary of serious periods (Medium+)
        serious = audit[audit['gap_category'].isin(['Medium','Long','VeryLong'])]
        if not serious.empty:
            print(f'  Serious periods (Medium+):')
            for _, r in serious.iterrows():
                print(f'    [{r.flag}] {str(r.start)[:13]} → {str(r.end)[:13]}  '
                      f'{r.duration_h}h ({r.gap_category})  '
                      f'score={r.avg_score}  {r.factors}')
        ALL_STUCK.append(audit)
    else:
        print(f'  No stuck periods detected')

# Combined file
if ALL_STUCK:
    combined = pd.concat(ALL_STUCK, ignore_index=True)
    path = os.path.join(OUTPUT_DIR, 'audit_stuck_all_stations.csv')
    combined.to_csv(path, index=False)
    print(f'\nSaved: audit_stuck_all_stations.csv ({len(combined)} total periods)')


CB01: 16 stuck period(s), 332 hours nulled
  Saved: audit_stuck_CB01.csv
  Serious periods (Medium+):
    [SUSPECT] 2022-11-23 17 → 2022-11-26 09  65h (Medium)  score=3.37  RH_std=0.123<2.0
    [SUSPECT] 2023-01-30 15 → 2023-02-03 07  89h (Medium)  score=3.02  RH_std=0.185<2.0
    [SUSPECT] 2023-11-10 07 → 2023-11-11 16  34h (Medium)  score=3.0  RH_std=0.000<2.0
    [SUSPECT] 2024-01-23 17 → 2024-01-25 11  43h (Medium)  score=3.0  RH_std=0.000<2.0
CB04: 95 stuck period(s), 1,923 hours nulled
  Saved: audit_stuck_CB04.csv
  Serious periods (Medium+):
    [SUSPECT] 2014-12-31 05 → 2015-01-03 11  79h (Medium)  score=3.2  RH_std=1.064<2.0
    [SUSPECT] 2015-02-23 07 → 2015-02-25 11  53h (Medium)  score=3.23  RH_std=0.291<2.0 | RH_Temp_corr=0.135<0.177
    [SUSPECT] 2015-02-28 13 → 2015-03-03 13  73h (Medium)  score=3.03  RH_std=0.000<2.0
    [SUSPECT] 2015-03-21 04 → 2015-03-22 11  32h (Medium)  score=3.0  RH_std=0.901<2.0
    [SUSPECT] 2015-11-28 07 → 2015-11-30 12  54h (Medium)  score=3.

## Stage 5 — Gap catalog
Catalogs every run of missing RH values (post stuck-sensor nulling) by duration.

**Writes:** `{station}_gap_summary.csv`, `audit_gap_catalog_all.csv`.


In [13]:
# ── Stage 5: Gap catalog ─────────────────────────────────────────────────────
ALL_GAPS = []

for sid, df in VALID_CLEAN.items():
    if COL_RH not in df.columns: continue
    rh    = df[COL_RH]
    is_na = rh.isna()
    runs  = (is_na != is_na.shift()).cumsum()
    gaps  = []
    for rid, grp in rh[is_na].groupby(runs[is_na]):
        h = len(grp)
        gaps.append({'station':sid,'start':grp.index[0],'end':grp.index[-1],
                     'duration_h':h,'category':categorize_gap_duration(h)})
    df_gaps = pd.DataFrame(gaps)
    if not df_gaps.empty:
        path = os.path.join(OUTPUT_DIR, f'{sid}_gap_summary.csv')
        df_gaps.to_csv(path, index=False)
        ALL_GAPS.append(df_gaps)
        cats = df_gaps['category'].value_counts().to_dict()
        print(f'  {sid}: {len(df_gaps)} gaps — {cats}')
    else:
        print(f'  {sid}: no gaps')

if ALL_GAPS:
    pd.concat(ALL_GAPS, ignore_index=True).to_csv(
        os.path.join(OUTPUT_DIR, 'audit_gap_catalog_all.csv'), index=False)
    print('Saved: audit_gap_catalog_all.csv')


  CB01: 16 gaps — {'Short': 12, 'Medium': 4}
  CB04: 99 gaps — {'Short': 76, 'Medium': 23}
  CB06: 131 gaps — {'Short': 89, 'Medium': 37, 'VeryLong': 3, 'Long': 2}
  FD02: 101 gaps — {'Short': 77, 'Medium': 21, 'Long': 2, 'VeryLong': 1}
  FD03: 158 gaps — {'Short': 99, 'Medium': 48, 'Long': 11}
  WC05: 145 gaps — {'Short': 113, 'Medium': 29, 'VeryLong': 2, 'Long': 1}
Saved: audit_gap_catalog_all.csv


## Stage 6 — Monthly averages
Monthly RH means, computed only where at least `MIN_MONTH_COVERAGE` of the month's
hours have valid data; otherwise the month is `NaN`.

**Writes:** `{station}_rh_monthly_clean.csv`.


In [14]:
# ── Stage 6: Monthly averages ────────────────────────────────────────────────
def _monthly_agg(group, min_coverage=MIN_MONTH_COVERAGE):
    yr  = group.index.year[0]
    mo  = group.index.month[0]
    exp = calendar.monthrange(yr, mo)[1] * 24
    valid = group.dropna()
    return valid.mean() if len(valid)/exp >= min_coverage else np.nan

MONTHLY_CLEAN = {}

for sid, df in VALID_CLEAN.items():
    if COL_RH not in df.columns: continue
    rh = df[COL_RH]

    monthly = rh.groupby([rh.index.year, rh.index.month]).apply(_monthly_agg)
    monthly.index.names = ['year','month']
    pivot   = monthly.unstack('month').round(2)
    pivot.columns = [MONTHS[m-1] for m in pivot.columns]
    pivot.index.name = 'year'
    MONTHLY_CLEAN[sid] = pivot

    path = os.path.join(OUTPUT_DIR, f'{sid}_rh_monthly_clean.csv')
    pivot.to_csv(path)
    print(f'  {sid}: {pivot.notna().sum().sum()} valid station-months saved')

print('Stage 6 complete.')


  CB01: 21 valid station-months saved
  CB04: 114 valid station-months saved
  CB06: 93 valid station-months saved
  FD02: 130 valid station-months saved
  FD03: 98 valid station-months saved
  WC05: 76 valid station-months saved
Stage 6 complete.


## Stage 7 — Jump detection
Two signals: hourly |ΔRH| within each station, and monthly within-year + cross-year.
All flags written to CSV.

**Writes:** `audit_hourly_jumps.csv`, `audit_monthly_within_year_jumps.csv`,
`audit_monthly_cross_year_anomalies.csv`.


In [15]:
# ── Stage 7: Jump detection ──────────────────────────────────────────────────

# ── 7a. Hourly jump threshold (data-derived) ──────────────────────────────────
all_hourly_deltas = []
for sid, df in VALID_CLEAN.items():
    if COL_RH in df.columns:
        delta = df[COL_RH].diff().abs().dropna()
        all_hourly_deltas.extend(delta.values)

hourly_thresh = np.percentile(
    [v for v in all_hourly_deltas if not np.isnan(v)], HOURLY_JUMP_PCT)
print(f'Hourly jump threshold ({HOURLY_JUMP_PCT}th pct): {hourly_thresh:.2f}%')

# ── 7b. Hourly jump flags ─────────────────────────────────────────────────────
all_hourly_jumps = []
for sid, df in VALID_CLEAN.items():
    if COL_RH not in df.columns: continue
    delta   = df[COL_RH].diff().abs()
    flagged = delta[delta >= hourly_thresh].dropna()
    if not flagged.empty:
        jdf = pd.DataFrame({
            'station':    sid,
            'timestamp':  flagged.index,
            'rh_before':  df[COL_RH].shift(1)[flagged.index].round(2).values,
            'rh_after':   df[COL_RH][flagged.index].round(2).values,
            'delta_rh':   flagged.round(2).values,
            'threshold':  hourly_thresh,
        })
        all_hourly_jumps.append(jdf)
        print(f'  {sid}: {len(jdf)} hourly jumps >= {hourly_thresh:.1f}%')

if all_hourly_jumps:
    combined = pd.concat(all_hourly_jumps, ignore_index=True)
    path = os.path.join(OUTPUT_DIR, 'audit_hourly_jumps.csv')
    combined.to_csv(path, index=False)
    print(f'Saved: audit_hourly_jumps.csv ({len(combined)} rows)')

# ── 7c. Monthly within-year jump flags ────────────────────────────────────────
all_monthly_jumps = []
all_cross_year    = []

for sid, df in MONTHLY_CLEAN.items():
    # Within-year
    for year, row in df.iterrows():
        for i in range(1, 12):
            pv, cv = row[MONTHS[i-1]], row[MONTHS[i]]
            if pd.isna(pv) or pd.isna(cv): continue
            delta = cv - pv
            thresh = WITHIN_THRESH[MONTH_LABELS[i]]
            if abs(delta) >= thresh:
                all_monthly_jumps.append({
                    'station':    sid, 'year': year,
                    'transition': f'{MONTH_LABELS[i-1]}→{MONTH_LABELS[i]}',
                    'from_val':   round(pv,2), 'to_val': round(cv,2),
                    'delta':      round(delta,2), 'threshold': thresh,
                })
    # Cross-year
    stats = df.agg(['mean','std'], axis=0).T
    for year, row in df.iterrows():
        for mi, mon in enumerate(MONTHS):
            v = row[mon]
            if pd.isna(v): continue
            mu = stats.loc[mon,'mean']; sd = stats.loc[mon,'std']
            if pd.isna(sd) or sd == 0: continue
            z = (v - mu) / sd
            if abs(z) >= CROSS_YEAR_Z:
                all_cross_year.append({
                    'station':    sid, 'year': year,
                    'month':      MONTH_LABELS[mi], 'value': round(v,2),
                    'month_mean': round(mu,2), 'month_sd': round(sd,2),
                    'z_score':    round(z,2),
                    'direction':  'Low' if v < mu else 'High',
                })

for data, fname, label in [
    (all_monthly_jumps, 'audit_monthly_within_year_jumps.csv', 'monthly within-year jumps'),
    (all_cross_year,    'audit_monthly_cross_year_anomalies.csv', 'cross-year anomalies'),
]:
    if data:
        combined = pd.DataFrame(data)
        path = os.path.join(OUTPUT_DIR, fname)
        combined.to_csv(path, index=False)
        print(f'Saved: {fname} ({len(combined)} rows)')

print('Stage 7 complete.')


Hourly jump threshold (95th pct): 14.03%
  CB01: 496 hourly jumps >= 14.0%
  CB04: 2844 hourly jumps >= 14.0%
  CB06: 5461 hourly jumps >= 14.0%
  FD02: 4819 hourly jumps >= 14.0%
  FD03: 3349 hourly jumps >= 14.0%
  WC05: 2169 hourly jumps >= 14.0%
Saved: audit_hourly_jumps.csv (19138 rows)
Saved: audit_monthly_within_year_jumps.csv (11 rows)
Saved: audit_monthly_cross_year_anomalies.csv (19 rows)
Stage 7 complete.


## Stage 8 — Cross-station distance & correlation (needs ≥2 stations)

Runs once all six `.dat` files are in `DAT_DIR` and Stages 1–7 have processed
each of them — this cell just reads back what's already in `MONTHLY_CLEAN` and
`STATION_COORDS`, so it's fine to re-run alone once more stations are added.

**Two separate matrices, deliberately not blended into one score:**
- **Physical distance** (haversine, km) — from coordinates only.
- **Data correlation** (Pearson, on overlapping monthly-mean RH) — from actual
  measurements.

Why keep them separate: nearby stations pull from the same weather systems, so
they *will* look similar on average — but elevation, canopy cover, and nearby
water sources can make two close stations behave quite differently, and two far
stations behave quite similarly. A station's best donor for gap-filling is the
one it's most *correlated* with, not necessarily the one it's closest to. Use
the distance matrix for context (does the correlation ranking make physical
sense?), and the correlation matrix for actually picking donors. Rows/pairs
where a close station has weak correlation, or a far station has strong
correlation, are flagged explicitly below — those are the cases worth a second
look before trusting either signal.


In [16]:
# ── Stage 8a: Station coordinates ────────────────────────────────────────────
# Confirmed against logger IDs: the numeric suffix in each logger ID matches
# the station code's suffix (e.g. FD03 -> CR1000-3), so this mapping is
# cross-checked, not just assumed from file order.
STATION_COORDS = {
    # station: (lat, lon, logger_id)
    'WC05': (30.3989, -98.6105, 'CR1000-5'),
    'CB01': (30.4193, -98.8046, 'CR200-26 (CR1000-1)'),
    'CB06': (30.4421, -98.8427, 'CR1000-6'),
    'CB04': (30.4600, -98.9407, 'CR1000-4'),
    'FD02': (30.2454, -98.7059, 'CR1000-2'),
    'FD03': (30.2758, -98.7242, 'CR1000-3'),
}

MIN_OVERLAP_MONTHS = 12  # minimum overlapping station-months to trust a correlation

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlmb = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

present_stations = [s for s in STATION_COORDS if s in MONTHLY_CLEAN]
if len(present_stations) < 2:
    print(f'Only {len(present_stations)} station(s) with monthly data available '
          f'({present_stations}) — need at least 2 to compare. Add more .dat '
          f'files to DAT_DIR and re-run Stages 1-7, then just re-run this cell.')
else:
    # ── Distance matrix ──────────────────────────────────────────────────────
    dist = pd.DataFrame(index=present_stations, columns=present_stations, dtype=float)
    for a in present_stations:
        for b in present_stations:
            lat1, lon1, _ = STATION_COORDS[a]
            lat2, lon2, _ = STATION_COORDS[b]
            dist.loc[a, b] = round(haversine_km(lat1, lon1, lat2, lon2), 2)
    dist.to_csv(os.path.join(OUTPUT_DIR, 'distance_matrix_km.csv'))
    print('Distance matrix (km):')
    print(dist)

    # ── Correlation matrix (monthly mean RH, overlapping months only) ────────
    long_frames = []
    for sid in present_stations:
        df = MONTHLY_CLEAN[sid][MONTHS].copy()
        long = df.stack()
        long.index = [f'{y}-{m}' for y, m in long.index]
        long_frames.append(long.rename(sid))
    monthly_wide = pd.concat(long_frames, axis=1)

    corr = pd.DataFrame(index=present_stations, columns=present_stations, dtype=float)
    overlap_n = pd.DataFrame(index=present_stations, columns=present_stations, dtype=int)
    for a in present_stations:
        for b in present_stations:
            pair = monthly_wide[[a, b]].dropna()
            overlap_n.loc[a, b] = len(pair)
            if a == b:
                corr.loc[a, b] = 1.0
            elif len(pair) >= MIN_OVERLAP_MONTHS:
                corr.loc[a, b] = round(pair[a].corr(pair[b]), 3)
            else:
                corr.loc[a, b] = np.nan  # not enough overlap to trust

    corr.to_csv(os.path.join(OUTPUT_DIR, 'correlation_matrix_monthly_rh.csv'))
    overlap_n.to_csv(os.path.join(OUTPUT_DIR, 'correlation_matrix_overlap_months.csv'))
    print(f'\nCorrelation matrix (monthly mean RH, min {MIN_OVERLAP_MONTHS} overlapping months):')
    print(corr)

    # ── Donor candidate table: distance rank vs correlation rank, side by side ──
    rows = []
    for a in present_stations:
        others = [s for s in present_stations if s != a]
        d_rank = dist.loc[a, others].rank().astype(int)
        c_rank = corr.loc[a, others].rank(ascending=False)
        for b in others:
            rows.append({
                'station': a, 'candidate': b,
                'distance_km': dist.loc[a, b], 'distance_rank': int(d_rank[b]),
                'correlation': corr.loc[a, b],
                'correlation_rank': int(c_rank[b]) if pd.notna(c_rank[b]) else None,
                'overlap_months': int(overlap_n.loc[a, b]),
            })
    donors = pd.DataFrame(rows)

    # Flag disagreements: nearest-by-distance isn't top-ranked by correlation, or vice versa
    donors['rank_gap'] = (donors['distance_rank'] - donors['correlation_rank']).abs()
    flagged = donors[(donors['distance_rank'] == 1) & (donors['correlation_rank'] > 2)]
    donors.to_csv(os.path.join(OUTPUT_DIR, 'donor_candidates_all.csv'), index=False)

    print(f'\nSaved: distance_matrix_km.csv, correlation_matrix_monthly_rh.csv, '
          f'donor_candidates_all.csv')
    if not flagged.empty:
        print(f'\n⚠ Closest station is NOT the best-correlated donor for:')
        for _, r in flagged.iterrows():
            print(f'  {r.station}: nearest is {r.candidate} ({r.distance_km}km) but '
                  f'correlation rank is #{r.correlation_rank} '
                  f'(corr={r.correlation}, n={r.overlap_months} months)')
    else:
        print('\nNo major distance/correlation disagreements flagged.')


Distance matrix (km):
       WC05   CB01   CB06   CB04   FD02   FD03
WC05   0.00  18.75  22.78  32.38  19.37  17.51
CB01  18.75   0.00   4.45  13.81  21.53  17.72
CB06  22.78   4.45   0.00   9.60  25.51  21.71
CB04  32.38  13.81   9.60   0.00  32.82  29.17
FD02  19.37  21.53  25.51  32.82   0.00   3.81
FD03  17.51  17.72  21.71  29.17   3.81   0.00

Correlation matrix (monthly mean RH, min 12 overlapping months):
       WC05   CB01   CB06   CB04   FD02   FD03
WC05  1.000    NaN  0.188  0.672  0.679  0.187
CB01    NaN  1.000  0.727  0.935  0.942  0.986
CB06  0.188  0.727  1.000  0.560  0.694  0.157
CB04  0.672  0.935  0.560  1.000  0.919  0.420
FD02  0.679  0.942  0.694  0.919  1.000  0.475
FD03  0.187  0.986  0.157  0.420  0.475  1.000

Saved: distance_matrix_km.csv, correlation_matrix_monthly_rh.csv, donor_candidates_all.csv

⚠ Closest station is NOT the best-correlated donor for:
  WC05: nearest is FD03 (17.51km) but correlation rank is #4.0 (corr=0.187, n=52 months)
  CB01: nearest 

### Optional — station elevation

Elevation isn't derivable from lat/lon alone. This cell queries the free
[Open-Elevation](https://open-elevation.com) API (needs internet, which Colab
has) as a best-effort lookup — it's a community-run service and can be slow or
occasionally unavailable, so treat failures as "skip it," not "pipeline broken."
Elevation and land cover near each station are exactly the kind of thing that
can make two close-by stations diverge, so this is meant to sit alongside the
distance/correlation tables above, not replace them.


In [17]:
import requests

try:
    locations = '|'.join(f'{lat},{lon}' for lat, lon, _ in STATION_COORDS.values())
    resp = requests.get('https://api.open-elevation.com/api/v1/lookup',
                        params={'locations': locations}, timeout=15)
    resp.raise_for_status()
    elevations = [r['elevation'] for r in resp.json()['results']]
    elev_df = pd.DataFrame({
        'station': list(STATION_COORDS.keys()),
        'elevation_m': elevations,
    })
    elev_df.to_csv(os.path.join(OUTPUT_DIR, 'station_elevations.csv'), index=False)
    print(elev_df.to_string(index=False))
except Exception as e:
    print(f'Elevation lookup skipped ({e}). Not required for the pipeline to work.')


station  elevation_m
   WC05        503.0
   CB01        567.0
   CB06        536.0
   CB04        590.0
   FD02        457.0
   FD03        499.0


## Output reference

Every file below now lives in `OUTPUT_DIR`. The dashboard notebook
(`txson_met_pipeline_2_dashboard.ipynb`) reads `clean_*`, `*_rh_monthly_clean.csv`,
`valid_*`/`flagged_*`, and the `audit_*` files directly — no need to re-run this
notebook to explore results, only to regenerate them.

| File | Stage | Contents |
|---|---|---|
| `{station}_met_prewashed.csv` | 2 | Full prewashed series, original column names |
| `{station}_met_prewashed_<step>_report.csv` | 2 | Rows changed by each prewash step |
| `valid_{station}_met.csv` | 3 | Rows with RH in `[RH_MIN, RH_MAX]` |
| `flagged_{station}_met.csv` | 3 | Rows with out-of-range RH |
| `clean_{station}_met.csv` | 4 | `valid_*` with stuck-sensor periods nulled — **`VALID_CLEAN`** |
| `audit_stuck_{station}.csv` / `audit_stuck_all_stations.csv` | 4 | Stuck/suspect period audit log |
| `{station}_gap_summary.csv` / `audit_gap_catalog_all.csv` | 5 | Missing-data run catalog |
| `{station}_rh_monthly_clean.csv` | 6 | Monthly RH pivot (year × month) — **`MONTHLY_CLEAN`** |
| `audit_hourly_jumps.csv` | 7 | Hour-to-hour RH jumps ≥ 95th-pct threshold |
| `audit_monthly_within_year_jumps.csv` | 7 | Month-to-month jumps within a year |
| `audit_monthly_cross_year_anomalies.csv` | 7 | Same-month, cross-year z-score outliers |
| `distance_matrix_km.csv` | 8 | Haversine distance between all station pairs |
| `correlation_matrix_monthly_rh.csv` | 8 | Pearson correlation on overlapping monthly RH |
| `donor_candidates_all.csv` | 8 | Distance rank vs. correlation rank per station pair |
| `station_elevations.csv` | 8 (optional) | Best-effort elevation lookup |

**Next step:** open `txson_met_pipeline_2_dashboard.ipynb`, point it at this same
`OUTPUT_DIR`, and run it to explore the results interactively.
